# SmartPantry AI: Mochi AI Assistant

## Phase 4 - AI Assistant Development and Testing

**Project:** SmartPantry AI  
**Component:** Mochi AI Assistant  
**Python Module:** `modules/ai_assistant.py`  
**Notebook:** `notebooks/ai_assistant.ipynb`

### Overview

Mochi is the conversational AI assistant for SmartPantry AI. Its purpose is to help users interact with food and pantry information through natural language.

This notebook documents the assistant's implementation, functionality, testing, and planned integration with other SmartPantry components.

### Phase 4 Objectives

- Examine the existing AI assistant implementation.
- Test assistant responses using controlled inputs.
- Evaluate how Mochi handles errors and unavailable information.
- Explore integration with pantry inventory, recipe recommendations, storage guidance, and food recall information.
- Document test results, limitations, and future improvements.

### Testing Approach

Initial tests will use mocked or controlled responses to avoid unnecessary paid API requests. Live API testing will be performed only when necessary and clearly identified.

Food safety information must be handled carefully. Mochi should not invent recall results, expiration dates, or storage recommendations.


### Google Colab and Local Jupyter Setup

This notebook supports Google Colab and local Jupyter environments.

**Google Colab**

Clone the SmartPantry AI repository and install dependencies:

```python
!git clone https://github.com/SmartPantry-AI/smartpantry-ai.git
%cd smartpantry-ai
%pip install -r requirements.txt
```

**Local Jupyter / VS Code**

Open the SmartPantry AI repository and select the project's Python environment.

Install dependencies if needed:

```bash
python -m pip install -r requirements.txt
```

**Testing Notes**

- Automated tests use mocked OpenAI responses to avoid paid API calls.
- Live API testing requires valid credentials.
- Supabase integration tests must distinguish mocked operations from real database operations.
- Final submission should use a verified Git commit so the notebook and Python modules match.


In [1]:

import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    project_root = Path("/content/smartpantry-ai")

    if not project_root.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--branch",
                "main",
                "https://github.com/SmartPantry-AI/smartpantry-ai.git",
                str(project_root),
            ],
            check=True,
        )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-r",
            str(project_root / "requirements.txt"),
        ],
        check=True,
    )

else:
    project_root = Path.cwd().resolve()

    if not (project_root / "modules").is_dir():
        project_root = project_root.parent

assert (project_root / "modules" / "ai_assistant.py").is_file(), (
    "SmartPantry project folder not found."
)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Environment:", "Google Colab" if IN_COLAB else "Local")
print("SmartPantry project path configured.")
print("Project root:", project_root)


Environment: Local
SmartPantry project path configured.
Project root: C:\Users\Samir\GitHub\smartpantry-ai


In [3]:

import os

os.chdir(project_root)

print("Working directory:", os.getcwd())

from modules.ai_assistant import format_pantry_inventory

print("PASS - Mochi module imported successfully")


Working directory: C:\Users\Samir\GitHub\smartpantry-ai
PASS - Mochi module imported successfully


## 1. Existing AI Assistant Implementation

The current Mochi assistant is implemented in
`modules/ai_assistant.py`.


### Main Functions

Mochi's current implementation is located in `modules/ai_assistant.py`.

- `ask_ai()`: Sends questions to OpenAI using Mochi's instructions and supports conversation history.
- `interpret_pantry_request()`: Converts natural-language requests into structured pantry commands and validates their contents.
- `handle_pantry_command()`: Executes supported commands using the in-memory `PantryManager`.
- `confirm_pantry_command()`: Prevents in-memory pantry changes until the user confirms the command.
- `handle_supabase_pantry_command()`: Adds a validated pantry item through the Supabase pantry module using a supplied client.
- `format_pantry_inventory()`: Formats saved pantry records into readable text for Mochi.

### Important Implementation Distinction

Mochi supports both an earlier in-memory pantry workflow and a newer Supabase pantry workflow.

The in-memory workflow includes a confirmation function. The Supabase command function does not independently request confirmation, so the application must enforce confirmation before calling it.

Automated tests verify these functions using controlled inputs and mocks. Live database persistence and the complete Streamlit interaction require separate integration verification.



### Current Capabilities

The current Mochi implementation supports:

1. **Conversational assistance:** Responds to questions using OpenAI and Mochi's SmartPantry-specific instructions.
2. **Conversation history:** Accepts previous messages to provide context for follow-up questions.
3. **Structured pantry commands:** Interprets requests to add pantry items and validates ingredients, quantities, and measurement units.
4. **User confirmation:** Supports confirmation before executing commands in the in-memory pantry workflow.
5. **Supabase pantry integration:** Provides a function for adding items through the Supabase pantry module using a supplied client.
6. **Inventory responses:** Formats saved pantry records into readable responses, including empty pantries and separate item batches.
7. **Error handling:** Returns a user-friendly fallback response when the conversational OpenAI request fails.

### Verification Status

These capabilities are implemented in `modules/ai_assistant.py` and have corresponding automated tests in `tests/test_ai_assistant.py`.

The test file currently defines 20 test functions representing 25 pytest cases, including six parameterized invalid-command scenarios.

These counts describe the existing automated test coverage. A fresh test execution is required to confirm the current passing results.

The automated tests do not independently establish successful live OpenAI requests, real Supabase persistence, or complete Streamlit integration.


### Observations

1. The assistant uses `gpt-4.1-mini`.
2. Pantry commands currently support the `add` action.
3. A confirmation function is available before executing pantry commands.
4. The general conversational function does not automatically retrieve live pantry inventory.
5. The quantity field is defined as a string in the structured response schema, but subsequent validation requires a numeric value.
6. Additional integration is needed to connect conversational responses with storage guidance, recipe recommendations, and recall checking.


### Phase 4 Development Focus

Phase 4 focuses on evaluating Mochi's existing conversational and pantry-management capabilities, including its integration with Supabase.

The evaluation covers structured command validation, user confirmation, inventory formatting, conversation history, and error handling.

Particular attention is given to the differences between the in-memory and Supabase workflows, ensuring that pantry modifications require appropriate user confirmation.

Additional integration testing is needed to verify how Mochi interacts with the Streamlit application, authenticated pantry records, storage guidance, recipe recommendations, and food recall information.

The original notebook test results are preserved as historical development evidence. Updated automated test results will be documented separately after executing the current test suite.



In [4]:
import sys
from pathlib import Path

current_directory = Path.cwd()

if (current_directory / "modules").is_dir():
    project_root = current_directory
elif (current_directory.parent / "modules").is_dir():
    project_root = current_directory.parent
elif (current_directory / "smartpantry-ai" / "modules").is_dir():
    project_root = current_directory / "smartpantry-ai"
else:
    raise FileNotFoundError("SmartPantry project folder not found.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("SmartPantry project path configured.")
print("Repository exists:", project_root.exists())
print("Project root:", project_root)

SmartPantry project path configured.
Repository exists: True
Project root: C:\Users\Samir\GitHub\smartpantry-ai


In [5]:
from modules import ai_assistant

print("Mochi module imported successfully!")

Mochi module imported successfully!


In [6]:
from pathlib import Path

module_path = Path(ai_assistant.__file__)
module_code = module_path.read_text(encoding="utf-8")

print("Module location:", module_path)
print(
    "Quantity schema corrected:",
    '"quantity": {"type": "number"}' in module_code
)

Module location: C:\Users\Samir\GitHub\smartpantry-ai\modules\ai_assistant.py
Quantity schema corrected: True


In [7]:
# Test 1: Reproduce the original quantity type mismatch

original_schema_type = "string"
original_quantity = "2"

# Reproduce the original numeric validation condition.
is_valid_quantity = (
    isinstance(original_quantity, (int, float))
    and not isinstance(original_quantity, bool)
)

print("Original schema type:", original_schema_type)
print("Simulated quantity:", repr(original_quantity))
print("Python quantity type:", type(original_quantity).__name__)
print("Accepted by original validation:", is_valid_quantity)

assert original_schema_type == "string"
assert is_valid_quantity is False

print("Test 1: PASS - Original mismatch reproduced")

Original schema type: string
Simulated quantity: '2'
Python quantity type: str
Accepted by original validation: False
Test 1: PASS - Original mismatch reproduced



## 2. AI Assistant Testing

### Test 1: Original Pantry Quantity Validation Mismatch

**Test Type:** Historical bug reproduction / Regression test

**Function Investigated:** `interpret_pantry_request()`

**Objective:** Reproduce the original mismatch between the JSON response schema and Python quantity validation.

**Original Test Input:** "Add 2 cartons of milk to my pantry"

**Original Implementation:**
- JSON schema quantity type: `string`
- Simulated quantity: `"2"`
- Python validation required an `int` or `float`

**Original Failure Observed:**

```text
Validation error: Quantity must be a positive, finite numhber.
```

**Updated Reproduction Test Output:**

```text
Original schema type: string
Simulated quantity: '2'
Python quantity type: str
Accepted by original validation: False
Test 1: PASS - Original mismatch reproduced
```

**Status:** PASS (historical mismatch successfully reproduced)

**Root Cause:** The original JSON schema requested a string, but Python validation required a numeric quantity.

**Correction:** Changed the quantity schema from `string` to `number` in `modules/ai_assistant.py`.

**API Usage:** None.

**Database Usage:** None.

**Conclusion:** The test independently reproduces the original data-type mismatch. The historical failure is preserved as evidence, while the current executable test passes.

**Limitation:** This reproduction tests the original validation condition independently rather than executing the original version of `interpret_pantry_request()`. Test 2 verifies the corrected module.


In [8]:
import importlib

importlib.reload(ai_assistant)

print("Mochi module reloaded successfully!")

Mochi module reloaded successfully!


In [9]:
from unittest.mock import patch, MagicMock

mock_response = MagicMock()
mock_response.output_text = """
{
    "action": "add",
    "ingredient": "milk",
    "quantity": 2,
    "unit": "cartons"
}
"""

with patch("modules.ai_assistant.OpenAI") as mock_openai:
    mock_openai.return_value.responses.create.return_value = mock_response

    with patch.dict("os.environ", {"OPENAI_API_KEY": "test-key"}):
        try:
            result = ai_assistant.interpret_pantry_request(
                "Add 2 cartons of milk to my pantry"
            )
            print("Result:", result)

            # Verify that the corrected schema requests a number
            call_args = mock_openai.return_value.responses.create.call_args
            schema = call_args.kwargs["text"]["format"]["schema"]
            print("Schema quantity type:", schema["properties"]["quantity"]["type"])

        except ValueError as error:
            print("Validation error:", error)

Result: {'action': 'add', 'ingredient': 'milk', 'quantity': 2, 'unit': 'cartons'}
Schema quantity type: number


### Test 2: Pantry Quantity Validation Correction

**Test Type:** Mocked API response / Regression test

**Function:** `interpret_pantry_request()`

**Objective:** Verify that the corrected JSON schema accepts a positive numeric quantity.

**Correction Applied:**
Changed the quantity property in `modules/ai_assistant.py` from `"type": "string"` to `"type": "number"`.

**Test Input:** "Add 2 cartons of milk to my pantry"

**Simulated OpenAI Response:**
- Action: add
- Ingredient: milk
- Quantity: 2
- Unit: cartons

**Expected Result:** The function returns a valid structured pantry command, and the JSON schema specifies a numeric quantity.

**Actual Result:**
```text
Result: {'action': 'add', 'ingredient': 'milk', 'quantity': 2, 'unit': 'cartons'}
Schema quantity type: number

In [10]:
from unittest.mock import MagicMock

mock_pantry = MagicMock()

command = {
    "action": "add",
    "ingredient": "milk",
    "quantity": 2,
    "unit": "cartons"
}

result = ai_assistant.confirm_pantry_command(
    command,
    mock_pantry,
    confirmed=False
)

print("Result:", result)
print("Pantry add called:", mock_pantry.add_item.called)

assert result is None
assert mock_pantry.add_item.call_count == 0

print("Test 3: PASS")

Result: None
Pantry add called: False
Test 3: PASS


### Test 3: Pantry Addition Without User Confirmation

**Test Type:** Mocked unit test / Confirmation safeguard

**Function:** `confirm_pantry_command()`

**Objective:** Verify that Mochi does not modify pantry inventory when the user has not confirmed the requested action.

**Test Input:**
- Action: add
- Ingredient: milk
- Quantity: 2
- Unit: cartons
- User confirmation: False

**Expected Result:**
- The function returns `None`.
- The pantry manager's `add_item()` function is not called.
- No pantry modification occurs.

**Actual Result:**
```text
Result: None
Pantry add called: False
Test 3: PASS

In [11]:
from unittest.mock import MagicMock

mock_pantry = MagicMock()
mock_pantry.add_item.return_value = "Item added successfully"

command = {
    "action": "add",
    "ingredient": "milk",
    "quantity": 2,
    "unit": "cartons"
}

result = ai_assistant.confirm_pantry_command(
    command,
    mock_pantry,
    confirmed=True
)

print("Result:", result)
print("Pantry add called:", mock_pantry.add_item.called)

mock_pantry.add_item.assert_called_once_with(
    "milk", 2, "cartons"
)

print("Test 4: PASS")

Result: Item added successfully
Pantry add called: True
Test 4: PASS



### Test 4: Pantry Addition With User Confirmation

**Test Type:** Mocked unit test / Positive test

**Function:** `confirm_pantry_command()`

**Objective:** Verify that Mochi correctly passes an approved pantry addition command to the pantry manager.

**Test Input:**
- Action: add
- Ingredient: milk
- Quantity: 2
- Unit: cartons
- User confirmation: True

**Expected Result:**
- The pantry manager's `add_item()` method is called exactly once.
- The correct ingredient, quantity, and unit are passed.
- The function returns the pantry manager's result.

**Actual Result:**
```text
Result: Item added successfully
Pantry add called: True
Test 4: PASS
```

**Status:** PASS

**API Usage:** None.

**Database Usage:** None. A mocked pantry manager was used.

**Conclusion:** The confirmation function correctly delegates an approved pantry addition to the pantry manager.

**Limitation:** This test does not verify actual database persistence or the Streamlit confirmation interface.


In [12]:
from unittest.mock import MagicMock

mock_pantry = MagicMock()

invalid_command = {
    "action": "delete",
    "ingredient": "milk",
    "quantity": 2,
    "unit": "cartons"
}

try:
    ai_assistant.handle_pantry_command(
        invalid_command,
        mock_pantry
    )

    print("Test 5: FAIL - Unsupported action was accepted")

except ValueError as error:
    print("Validation error:", error)

    mock_pantry.add_item.assert_not_called()

    print("Test 5: PASS")

Validation error: Unsupported pantry action.
Test 5: PASS



### Test 5: Unsupported Pantry Action

**Test Type:** Mocked unit test / Negative test

**Function:** `handle_pantry_command()`

**Objective:** Verify that Mochi rejects unsupported pantry actions without executing a pantry addition.

**Test Input:**
- Action: delete
- Ingredient: milk
- Quantity: 2
- Unit: cartons

**Expected Result:**
- The function raises a `ValueError`.
- The pantry manager's `add_item()` method is not called.

**Actual Result:**
```text
Validation error: Unsupported pantry action.
Test 5: PASS
```

**Status:** PASS

**API Usage:** None.

**Database Usage:** None. A mocked pantry manager was used.

**Conclusion:** The function correctly rejected the unsupported delete action.

**Limitation:** This test verifies that `add_item()` was not called. It does not establish whether future deletion functionality is implemented safely.


In [13]:
from unittest.mock import patch, MagicMock

mock_response = MagicMock()

mock_response.output_text = """
{
    "action": "add",
    "ingredient": "",
    "quantity": 2,
    "unit": "cartons"
}
"""

with patch("modules.ai_assistant.OpenAI") as mock_openai:
    mock_openai.return_value.responses.create.return_value = mock_response

    with patch.dict("os.environ", {"OPENAI_API_KEY": "test-key"}):
        try:
            result = ai_assistant.interpret_pantry_request(
                "Add 2 cartons to my pantry"
            )

            print("Test 6: FAIL - Missing ingredient accepted")

        except ValueError as error:
            print("Validation error:", error)
            print("Test 6: PASS")

Validation error: Ingredient name must be a non-empty string.
Test 6: PASS



### Test 6: Missing Ingredient Validation

**Test Type:** Mocked API response / Negative test

**Function:** `interpret_pantry_request()`

**Objective:** Verify that Mochi rejects a pantry addition request when the ingredient name is missing.

**Test Input:** "Add 2 cartons to my pantry"

**Simulated OpenAI Response:**
- Action: add
- Ingredient: "" (empty string)
- Quantity: 2
- Unit: cartons

**Expected Result:** The function raises a `ValueError` because the ingredient name is missing.

**Actual Result:**

```text
Validation error: Ingredient name must be a non-empty string.
Test 6: PASS
```

**Status:** PASS

**API Usage:** None. The OpenAI response was mocked.

**Database Usage:** None.

**Conclusion:** The function successfully rejected an empty ingredient name.

**Limitation:** The test confirms validation of an empty ingredient field, but does not verify whether a live AI model might invent an ingredient when interpreting an incomplete request.


In [14]:
from unittest.mock import patch, MagicMock

mock_response = MagicMock()

mock_response.output_text = """
{
    "action": "add",
    "ingredient": "milk",
    "quantity": 0,
    "unit": "cartons"
}
"""

with patch("modules.ai_assistant.OpenAI") as mock_openai:
    mock_openai.return_value.responses.create.return_value = mock_response

    with patch.dict("os.environ", {"OPENAI_API_KEY": "test-key"}):
        try:
            result = ai_assistant.interpret_pantry_request(
                "Add 0 cartons of milk to my pantry"
            )

            print("Test 7: FAIL - Zero quantity accepted")

        except ValueError as error:
            print("Validation error:", error)
            print("Test 7: PASS")

Validation error: Quantity must be a positive, finite numhber.
Test 7: PASS



### Test 7: Zero Quantity Validation

**Test Type:** Mocked API response / Negative test

**Function:** `interpret_pantry_request()`

**Objective:** Verify that Mochi rejects pantry addition requests containing a quantity of zero.

**Test Input:** "Add 0 cartons of milk to my pantry"

**Simulated OpenAI Response:**
- Action: add
- Ingredient: milk
- Quantity: 0
- Unit: cartons

**Expected Result:** The function raises a `ValueError` because a pantry addition requires a positive quantity.

**Actual Result:**

```text
Validation error: Quantity must be a positive, finite numhber.
Test 7: PASS
```

**Status:** PASS

**API Usage:** None. The OpenAI response was mocked.

**Database Usage:** None.

**Conclusion:** The function successfully rejected a quantity of zero. This confirms that the positive-quantity validation works for this test case.

**Limitation:** Negative quantities, nonnumeric values, and non-finite numbers have not yet been tested.


In [15]:
from unittest.mock import patch, MagicMock

mock_response = MagicMock()
mock_response.output_text = """
{
    "action": "add",
    "ingredient": "milk",
    "quantity": -3,
    "unit": "cartons"
}
"""

with patch("modules.ai_assistant.OpenAI") as mock_openai:
    mock_openai.return_value.responses.create.return_value = mock_response

    with patch.dict("os.environ", {"OPENAI_API_KEY": "test-key"}):
        try:
            ai_assistant.interpret_pantry_request(
                "Add -3 cartons of milk to my pantry"
            )
        except ValueError as error:
            print("Validation error:", error)
            print("Test 8: PASS")
        else:
            print("Test 8: FAIL - Negative quantity accepted")

Validation error: Quantity must be a positive, finite numhber.
Test 8: PASS



### Test 8: Negative Quantity Validation

**Test Type:** Mocked API response / Negative test

**Function:** `interpret_pantry_request()`

**Objective:** Verify that Mochi rejects pantry addition requests containing a negative quantity.

**Test Input:** "Add -3 cartons of milk to my pantry"

**Simulated OpenAI Response:**
- Action: add
- Ingredient: milk
- Quantity: -3
- Unit: cartons

**Expected Result:** The function raises a `ValueError` because pantry additions require positive quantities.

**Actual Result:**

```text
Validation error: Quantity must be a positive, finite numhber.
Test 8: PASS
```

**Status:** PASS

**API Usage:** None. The OpenAI response was mocked.

**Database Usage:** None.

**Conclusion:** The function successfully rejected a negative quantity, confirming that the positive-quantity validation handles both zero and negative values.

**Limitation:** Nonnumeric quantities, non-finite values, and missing measurement units require further testing.


In [16]:
from unittest.mock import patch, MagicMock

mock_response = MagicMock()

mock_response.output_text = """
{
    "action": "add",
    "ingredient": "milk",
    "quantity": "two",
    "unit": "cartons"
}
"""

with patch("modules.ai_assistant.OpenAI") as mock_openai:
    mock_openai.return_value.responses.create.return_value = mock_response

    with patch.dict("os.environ", {"OPENAI_API_KEY": "test-key"}):
        try:
            ai_assistant.interpret_pantry_request(
                "Add two cartons of milk to my pantry"
            )

        except ValueError as error:
            print("Validation error:", error)
            print("Test 9: PASS")

        else:
            print("Test 9: FAIL - Nonnumeric quantity accepted")

Validation error: Quantity must be a positive, finite numhber.
Test 9: PASS



### Test 9: Nonnumeric Quantity Validation

**Test Type:** Mocked API response / Negative test

**Function:** `interpret_pantry_request()`

**Objective:** Verify that Mochi rejects a quantity represented as nonnumeric text instead of a valid number.

**Test Input:** "Add two cartons of milk to my pantry"

**Simulated OpenAI Response:**
- Action: add
- Ingredient: milk
- Quantity: "two"
- Unit: cartons

**Expected Result:** The function raises a `ValueError` because the quantity is not a numeric value.

**Actual Result:**

```text
Validation error: Quantity must be a positive, finite numhber.
Test 9: PASS
```

**Status:** PASS

**API Usage:** None. The OpenAI response was mocked.

**Database Usage:** None.

**Conclusion:** The function successfully rejected a nonnumeric quantity, preventing an improperly formatted value from being returned as a valid pantry command.

**Limitation:** This test intentionally supplied a response that violates the corrected JSON schema. It does not evaluate how reliably the live OpenAI model converts written numbers into numeric values.


In [17]:
from unittest.mock import patch, MagicMock

mock_response = MagicMock()

mock_response.output_text = """
{
    "action": "add",
    "ingredient": "milk",
    "quantity": 2,
    "unit": ""
}
"""

with patch("modules.ai_assistant.OpenAI") as mock_openai:
    mock_openai.return_value.responses.create.return_value = mock_response

    with patch.dict("os.environ", {"OPENAI_API_KEY": "test-key"}):
        try:
            ai_assistant.interpret_pantry_request(
                "Add 2 milk to my pantry"
            )

        except ValueError as error:
            print("Validation error:", error)
            print("Test 10: PASS")

        else:
            print("Test 10: FAIL - Missing unit accepted")

Validation error: Unit must be a non-empty string
Test 10: PASS



### Test 10: Missing Measurement Unit Validation

**Test Type:** Mocked API response / Negative test

**Function:** `interpret_pantry_request()`

**Objective:** Verify that Mochi rejects a pantry addition request when the measurement unit is missing.

**Test Input:** "Add 2 milk to my pantry"

**Simulated OpenAI Response:**
- Action: add
- Ingredient: milk
- Quantity: 2
- Unit: "" (empty string)

**Expected Result:** The function raises a `ValueError` because the measurement unit is missing.

**Actual Result:**

```text
Validation error: Unit must be a non-empty string
Test 10: PASS
```

**Status:** PASS

**API Usage:** None. The OpenAI response was mocked.

**Database Usage:** None.

**Conclusion:** The function successfully rejected an empty measurement unit.

**Limitation:** This test does not verify whether the live AI model asks the user to clarify a missing unit or whether it might invent one.




## 3. Phase 4 Extended Testing

The original ten test scenarios document Mochi's initial development and validation.

Phase 4 extends this evaluation to the current implementation in `modules/ai_assistant.py`, with particular attention to Supabase integration, inventory formatting, conversation history, and error handling.

### Extended Testing Approach

The additional tests use mocks and controlled inputs to verify behavior without requiring live OpenAI API calls or database modifications.

The corresponding automated tests are maintained in `tests/test_ai_assistant.py`.

The following sections distinguish between existing automated test coverage and results confirmed through a fresh execution.

### Test 11: Supabase Pantry Addition

**Test Type:** Mocked integration test

**Function:** `handle_supabase_pantry_command()`

**Objective:** Verify that a valid structured pantry command is passed to the Supabase pantry module.

**Expected Result:** A valid addition command invokes the database insertion function with the expected item information.

**Verification Status:** Pending fresh test execution.



## 3. Testing Summary and Phase 4 Findings

### Test Results

Ten test scenarios were documented for Mochi's AI assistant.

The initial test identified a mismatch between the JSON schema and Python quantity validation. The schema was corrected to use a numeric type, and subsequent mocked tests passed.

The tests covered:
- Positive quantity validation
- User confirmation requirements
- Approved pantry additions
- Unsupported pantry actions
- Missing ingredients
- Zero and negative quantities
- Nonnumeric quantities
- Missing measurement units

All tests were rerun successfully in VS Code after the correction. No live OpenAI API calls or real database modifications were required.

### Current Limitations

These tests verify individual functions using mocked responses. They do not establish that:
- Mochi can retrieve a user's actual pantry inventory.
- Pantry additions persist correctly in the database.
- The Streamlit interface consistently requires confirmation.
- The live AI model reliably interprets incomplete or ambiguous requests.
- Mochi can retrieve verified storage, recipe, or recall information.

### Phase 4 Next Steps

Future development will focus on connecting Mochi to persistent pantry inventory, implementing safe confirmation workflows, integrating existing SmartPantry modules, and testing conversational functionality.

### Conclusion

The testing process identified and corrected a quantity-schema mismatch and verified the assistant's existing pantry command validation and confirmation safeguards.

The module now has a documented baseline for future integration testing.



### Test 11: Supabase Pantry Addition

**Test Type:** Mocked integration test

**Function:** `handle_supabase_pantry_command()`

**Objective:** Verify that a valid structured pantry command is passed to the Supabase pantry module.

**Test Input:**
- Action: `add`
- Ingredient: `rice`
- Quantity: `2`
- Unit: `cups`
- Database client: Mock client
- Database response: Simulated pantry item record

**Expected Result:**
- Mochi recognizes the structured addition command.
- The Supabase pantry insertion function is called with the correct client, ingredient, quantity, and unit.
- The function returns the simulated database result.
- No actual Supabase records are created.

**Verification Status:** Pending fresh test execution.

**Notes:** This test uses `unittest.mock.patch` to replace the Supabase insertion function. It verifies integration behavior without requiring a live database connection or paid OpenAI API calls.


In [18]:

from unittest.mock import patch

from modules.ai_assistant import handle_supabase_pantry_command

command = {
    "action": "add",
    "ingredient": "rice",
    "quantity": 2,
    "unit": "cups",
}

fake_client = object()
fake_result = [{"id": 101, "name": "rice"}]

with patch(
    "modules.supabase_pantry.add_pantry_item",
    return_value=fake_result
) as mock_add:
    result = handle_supabase_pantry_command(
        command,
        fake_client
    )

    mock_add.assert_called_once_with(
        client=fake_client,
        name="rice",
        quantity=2,
        unit="cups"
    )

    assert result == fake_result

print("Test 11: PASS - Supabase pantry addition verified with mock")


Test 11: PASS - Supabase pantry addition verified with mock



### Test 12: Invalid Supabase Pantry Command Validation

**Test Type:** Parameterized validation test

**Function:** `handle_supabase_pantry_command()`

**Objective:** Verify that Mochi rejects invalid pantry commands before attempting to update Supabase.

**Test Inputs:**
1. Empty ingredient name
2. Negative quantity
3. Zero quantity
4. Empty measurement unit
5. Unsupported action (`delete`)
6. Missing command (`None`)

**Expected Result:**
- Each invalid command raises a `ValueError`.
- No database insertion is performed.
- Invalid pantry data is prevented from reaching Supabase.

**Verification Status:** Pending fresh notebook test execution.

**Notes:** This test extends the Phase 4 validation documentation and corresponds to the existing parameterized automated test in `tests/test_ai_assistant.py`. It does not require a live Supabase connection or paid OpenAI API calls.


In [19]:

from modules.ai_assistant import handle_supabase_pantry_command
from unittest.mock import patch

invalid_commands = [
    {"action": "add", "ingredient": "", "quantity": 2, "unit": "cups"},
    {"action": "add", "ingredient": "rice", "quantity": -2, "unit": "cups"},
    {"action": "add", "ingredient": "rice", "quantity": 0, "unit": "cups"},
    {"action": "add", "ingredient": "rice", "quantity": 2, "unit": ""},
    {"action": "delete", "ingredient": "rice", "quantity": 2, "unit": "cups"},
    None,
]

with patch("modules.supabase_pantry.add_pantry_item") as mock_add:
    for command in invalid_commands:
        try:
            handle_supabase_pantry_command(command, object())
        except ValueError:
            pass
        else:
            raise AssertionError(
                f"Expected ValueError for invalid command: {command}"
            )

    mock_add.assert_not_called()

print("Test 12: PASS - All 6 invalid Supabase commands rejected")


Test 12: PASS - All 6 invalid Supabase commands rejected



### Test 13: Pantry Inventory Formatting

**Test Type:** Unit test

**Function:** `format_pantry_inventory()`

**Objective:** Verify that Mochi correctly formats pantry inventory records into readable text.

**Test Input:**
- Banana: 5 item
- Rice: 10 lb

**Expected Result:**
- The formatted response includes `Banana: 5 item`.
- The formatted response includes `Rice: 10 lb`.
- Item names, quantities, and measurement units are preserved.
- No database or OpenAI API calls are required.

**Verification Status:** Pending full notebook execution.

**Notes:** This test documents the existing inventory formatting functionality in `tests/test_ai_assistant.py`. It uses sample pantry records and does not access a live Supabase database.


In [20]:

from modules.ai_assistant import format_pantry_inventory

pantry_items = [
    {"name": "Banana", "quantity": 5, "unit": "item"},
    {"name": "Rice", "quantity": 10, "unit": "lb"},
]

result = format_pantry_inventory(pantry_items)

assert "Banana: 5 item" in result
assert "Rice: 10 lb" in result

print("Test 13: PASS - Pantry inventory formatting verified")


Test 13: PASS - Pantry inventory formatting verified



### Test 14: Empty Pantry Handling

**Test Type:** Unit test

**Function:** `format_pantry_inventory()`

**Objective:** Verify that Mochi provides an appropriate response when the pantry contains no items.

**Test Input:**
- Empty pantry inventory: `[]`

**Expected Result:**
- Mochi returns `Your pantry is currently empty.`
- No errors occur when processing an empty inventory.
- No database or OpenAI API calls are required.

**Verification Status:** Pending full notebook execution.

**Notes:** This test documents the existing empty pantry handling functionality in `tests/test_ai_assistant.py`. It uses an empty list to simulate a pantry without stored items.


In [21]:

from modules.ai_assistant import format_pantry_inventory

result = format_pantry_inventory([])

assert result == "Your pantry is currently empty."

print("Test 14: PASS - Empty pantry handling verified")


Test 14: PASS - Empty pantry handling verified



### Test 15: Preserving Separate Pantry Batches

**Test Type:** Unit test

**Function:** `format_pantry_inventory()`

**Objective:** Verify that Mochi preserves separate pantry records for the same ingredient when formatting inventory information.

**Test Input:**
- Rice: 10 lb
- Rice: 2 cups

**Expected Result:**
- The formatted inventory includes `Rice: 10 lb`.
- The formatted inventory includes `Rice: 2 cups`.
- Both pantry records remain visible.
- Quantities with different measurement units are not combined.
- No database or OpenAI API calls are required.

**Verification Status:** Pending full notebook execution.

**Notes:** This test documents the existing `test_mochi_preserves_separate_pantry_batches()` in `tests/test_ai_assistant.py`. It supports SmartPantry's design of maintaining separate inventory records rather than automatically merging batches.


In [22]:

from modules.ai_assistant import format_pantry_inventory

pantry_items = [
    {"name": "Rice", "quantity": 10, "unit": "lb"},
    {"name": "Rice", "quantity": 2, "unit": "cups"},
]

result = format_pantry_inventory(pantry_items)

assert "Rice: 10 lb" in result
assert "Rice: 2 cups" in result

print("Test 15: PASS - Separate pantry batches preserved")


Test 15: PASS - Separate pantry batches preserved




### Test Coverage

| Test | Description | Status |
|---|---|---|
| 1–10 | Original Mochi functionality and validation tests | PASS |
| 11 | Supabase pantry addition using a mock | PASS |
| 12 | Invalid Supabase pantry command validation (6 cases) | PASS |
| 13 | Pantry inventory formatting | PASS |
| 14 | Empty pantry handling | PASS |
| 15 | Preserving separate pantry batches | PASS |


### Results

**Total Documented Tests:** 15

**Execution Environment:** Google Colab

**Tests Passed:** 15/15

**Tests Failed:** 0

**Additional Validation:** Test 12 rejected all 6 invalid Supabase commands.

All documented notebook tests passed during full execution in Google Colab. Phase 4 tests used mocked dependencies without making live OpenAI API calls or modifying Supabase records.



### Testing Methodology

Testing uses Python assertions and mocked dependencies to verify expected behavior without requiring live external services.

Mocking allows the notebook to simulate OpenAI responses and Supabase operations while avoiding unnecessary API calls or database modifications.

The tests focus on:
- Structured pantry command processing
- Input validation and error handling
- Supabase pantry integration
- Inventory formatting
- Preservation of separate pantry records


### Limitations

- Mocked tests do not verify live Supabase connectivity.
- Mocked OpenAI responses do not measure real-world language interpretation accuracy.
- Inventory formatting tests use predefined sample records.
- Additional end-to-end testing is required to validate Mochi's behavior within the deployed SmartPantry application.

### Conclusion

The expanded notebook provides a structured evaluation of Mochi's original AI assistant functionality and its Phase 4 pantry integration features.

The additional tests examine command validation, Supabase integration, and inventory presentation. Together, they help verify that Mochi can support pantry management operations while maintaining appropriate input validation and data handling.

All 15 documented tests passed during the complete Google Colab execution, confirming the expected behavior of Mochi's tested functions, including Phase 4 Supabase command handling, input validation, and pantry inventory formatting.


# Phase 5 - Pantry Storage Metadata Integration

## Overview

Phase 5 extends Mochi's pantry confirmation workflow
to support optional storage metadata while preserving
the existing AI-powered command interpretation.

The AI command extraction schema remains unchanged.
Storage information is supplied and confirmed by
the user rather than inferred by the language model.

### New Functionality

- Purchase and package expiration dates.
- Storage location.
- Food classification.
- Preparation state and preparation date.
- Package state and opening date.
- Thawing date.
- Storage metadata validation.
- Supabase persistence of confirmed metadata.
- Backward compatibility with existing pantry commands.

### Design Principles

- Preserve explicit user confirmation.
- Do not invent storage conditions or dates.
- Reject contradictory metadata.
- Preserve missing information as null values.
- Avoid unnecessary AI API calls.

### Testing Methodology

The following demonstrations use mocked dependencies
and fixed test data.

No live OpenAI API calls or Supabase database
modifications are required.

The preceding notebook sections document historical
development. This section validates the Phase 5
implementation.

## 1. Storage Metadata Forwarding

This test verifies that a confirmed pantry command
preserves its original ingredient, quantity, and unit
while forwarding all nine optional storage metadata
fields to the Supabase persistence function.

The database operation is mocked to prevent actual
database modifications.

In [23]:
from unittest.mock import MagicMock, patch
from modules import ai_assistant as mochi

confirmed_command = {
    "action": "add",
    "ingredient": "cheddar cheese",
    "quantity": 1,
    "unit": "item",
}

storage_metadata = {
    "purchase_date": "2026-11-15",
    "expiration_date": "2026-12-30",
    "storage_location": "REFRIGERATOR",
    "prepared_date": None,
    "food_classification": "REGULAR",
    "preparation_state": None,
    "package_state": "OPENED",
    "opened_date": "2026-11-24",
    "thawed_date": None,
}

mock_client = MagicMock()

with patch("modules.supabase_pantry.add_pantry_item") as mock_add:
    mock_add.return_value = {"id": 5001}

    metadata_result = mochi.handle_supabase_pantry_command(
        confirmed_command,
        mock_client,
        storage_metadata=storage_metadata,
    )

    forwarded_metadata = mock_add.call_args.kwargs

print("Forwarded pantry information:\n")

for key, value in forwarded_metadata.items():
    if key != "client":
        print(f"{key}: {value}")

print("\nMock persistence result:", metadata_result)

Forwarded pantry information:

name: cheddar cheese
quantity: 1
unit: item
purchase_date: 2026-11-15
expiration_date: 2026-12-30
storage_location: REFRIGERATOR
prepared_date: None
food_classification: REGULAR
preparation_state: None
package_state: OPENED
opened_date: 2026-11-24
thawed_date: None

Mock persistence result: {'id': 5001}


In [24]:
assert forwarded_metadata["client"] is mock_client
assert forwarded_metadata["name"] == "cheddar cheese"
assert forwarded_metadata["quantity"] == 1
assert forwarded_metadata["unit"] == "item"

for field, expected_value in storage_metadata.items():
    assert forwarded_metadata[field] == expected_value, (
        f"Incorrect forwarded value for {field}."
    )

assert metadata_result == {"id": 5001}

mock_add.assert_called_once()

print("PASS: Original pantry information (4/4 checks).")
print("PASS: Storage metadata forwarding (9/9 checks).")
print("PASS: Mock persistence result (1/1 check).")
print("PASS: Single persistence call (1/1 check).")

print("\nStorage metadata forwarding: 15/15 checks passed.")

PASS: Original pantry information (4/4 checks).
PASS: Storage metadata forwarding (9/9 checks).
PASS: Mock persistence result (1/1 check).
PASS: Single persistence call (1/1 check).

Storage metadata forwarding: 15/15 checks passed.


## 2. Invalid Storage Metadata Validation

This test verifies that Mochi rejects unsupported
storage metadata values before attempting to save
a pantry item.

Each scenario must raise a ValueError and prevent
the persistence function from being called.

The database operation remains mocked throughout
the validation process.

In [25]:
invalid_metadata_cases = [
    (
        "Invalid metadata type",
        ["REFRIGERATOR"],
        "Storage metadata must be a dictionary.",
    ),
    (
        "Invalid storage location",
        {"storage_location": "COUNTER"},
        "Invalid storage_location.",
    ),
    (
        "Invalid food classification",
        {"food_classification": "FROZEN"},
        "Invalid food_classification.",
    ),
    (
        "Invalid preparation state",
        {"preparation_state": "WASHED"},
        "Invalid preparation_state.",
    ),
    (
        "Invalid package state",
        {"package_state": "DAMAGED"},
        "Invalid package_state.",
    ),
]

for case_name, metadata, expected_error in invalid_metadata_cases:

    with patch("modules.supabase_pantry.add_pantry_item") as mock_add:

        try:
            mochi.handle_supabase_pantry_command(
                confirmed_command,
                mock_client,
                storage_metadata=metadata,
            )

        except ValueError as error:
            assert str(error) == expected_error

        else:
            raise AssertionError(
                f"{case_name}: Expected ValueError."
            )

        mock_add.assert_not_called()

    print(f"PASS: {case_name}")

print("\nInvalid storage metadata: 10/10 checks passed.")

PASS: Invalid metadata type
PASS: Invalid storage location
PASS: Invalid food classification
PASS: Invalid preparation state
PASS: Invalid package state

Invalid storage metadata: 10/10 checks passed.


## 3. Contradictory Storage Conditions

This test verifies that Mochi rejects incompatible
combinations of food classification and preparation
state, even when the individual values are valid.

Contradictory metadata must raise a ValueError
and prevent the persistence function from being called.

In [26]:
contradictory_cases = [
    ("COOKED", "WHOLE"),
    ("COOKED", "CUT"),
    ("COOKED", "RAW"),
    ("REGULAR", "COOKED"),
]

for classification, preparation in contradictory_cases:

    metadata = {
        "food_classification": classification,
        "preparation_state": preparation,
    }

    if classification == "COOKED":
        expected_error = (
            "For cooked food, select Cooked as the preparation "
            "state or leave it unspecified."
        )
    else:
        expected_error = (
            "Please select Cooked food / leftovers "
            "as the food classification."
        )

    with patch("modules.supabase_pantry.add_pantry_item") as mock_add:

        try:
            mochi.handle_supabase_pantry_command(
                confirmed_command,
                mock_client,
                storage_metadata=metadata,
            )

        except ValueError as error:
            assert str(error) == expected_error

        else:
            raise AssertionError(
                "Contradictory metadata was accepted."
            )

        mock_add.assert_not_called()

    print(f"PASS: {classification} + {preparation} rejected")

print("\nContradictory conditions: 8/8 checks passed.")

PASS: COOKED + WHOLE rejected
PASS: COOKED + CUT rejected
PASS: COOKED + RAW rejected
PASS: REGULAR + COOKED rejected

Contradictory conditions: 8/8 checks passed.


## 4. Date and Chronology Validation

This test verifies that Mochi validates storage dates
before forwarding pantry information for persistence.

Validation includes:

- ISO date formatting (YYYY-MM-DD).
- Valid calendar dates.
- Rejection of future purchase, preparation,
  opening, and thawing dates.
- Opening and thawing dates cannot precede
  the purchase date.

Package expiration dates may be in the future.

Invalid dates must raise a ValueError and prevent
the persistence function from being called.

In [27]:
from datetime import date, timedelta

future_date = (date.today() + timedelta(days=1)).isoformat()

invalid_date_cases = [
    (
        "Incorrect date format",
        {"purchase_date": "11/24/2026"},
        "Invalid purchase_date. Expected YYYY-MM-DD.",
    ),
    (
        "Impossible calendar date",
        {"purchase_date": "2026-02-30"},
        "Invalid purchase_date. Expected YYYY-MM-DD.",
    ),
    (
        "Future purchase date",
        {"purchase_date": future_date},
        "Purchase date cannot be in the future.",
    ),
    (
        "Future preparation date",
        {"prepared_date": future_date},
        "Prepared date cannot be in the future.",
    ),
    (
        "Future opening date",
        {"opened_date": future_date},
        "Opened date cannot be in the future.",
    ),
    (
        "Future thawing date",
        {"thawed_date": future_date},
        "Thawed date cannot be in the future.",
    ),
    (
        "Opening before purchase",
        {
            "purchase_date": "2026-11-24",
            "opened_date": "2026-11-23",
        },
        "Opening date cannot precede the purchase date.",
    ),
    (
        "Thawing before purchase",
        {
            "purchase_date": "2026-11-24",
            "thawed_date": "2026-11-23",
        },
        "Thawing date cannot precede the purchase date.",
    ),
]

for case_name, metadata, expected_error in invalid_date_cases:

    with patch("modules.supabase_pantry.add_pantry_item") as mock_add:

        try:
            mochi.handle_supabase_pantry_command(
                confirmed_command,
                mock_client,
                storage_metadata=metadata,
            )

        except ValueError as error:
            assert str(error) == expected_error

        else:
            raise AssertionError(
                f"{case_name}: Expected ValueError."
            )

        mock_add.assert_not_called()

    print(f"PASS: {case_name}")

print("\nDate and chronology validation: 16/16 checks passed.")

PASS: Incorrect date format
PASS: Impossible calendar date
PASS: Future purchase date
PASS: Future preparation date
PASS: Future opening date
PASS: Future thawing date
PASS: Opening before purchase
PASS: Thawing before purchase

Date and chronology validation: 16/16 checks passed.


## 5. Backward Compatibility

This test verifies that existing pantry commands
remain functional without storage metadata.

Three scenarios are evaluated:

- Storage metadata omitted entirely.
- Storage metadata explicitly set to None.
- Storage metadata provided as an empty dictionary.

Each scenario must preserve the original pantry
information and successfully invoke the mocked
persistence function without inventing metadata.

In [28]:
legacy_command = {
    "action": "add",
    "ingredient": "apple",
    "quantity": 2,
    "unit": "item",
}

legacy_cases = [
    ("Metadata omitted", {}),
    ("Metadata set to None", {"storage_metadata": None}),
    ("Empty metadata dictionary", {"storage_metadata": {}}),
]

for case_name, optional_arguments in legacy_cases:

    with patch("modules.supabase_pantry.add_pantry_item") as mock_add:

        mock_add.return_value = {"id": 5002}

        result = mochi.handle_supabase_pantry_command(
            legacy_command,
            mock_client,
            **optional_arguments,
        )

        mock_add.assert_called_once_with(
            client=mock_client,
            name="apple",
            quantity=2,
            unit="item",
        )

        assert result == {"id": 5002}

    print(f"PASS: {case_name}")

print("\nBackward compatibility: 6/6 checks passed.")

PASS: Metadata omitted
PASS: Metadata set to None
PASS: Empty metadata dictionary

Backward compatibility: 6/6 checks passed.


## 6. Automated Mochi Regression Testing

In addition to the functional demonstrations,
SmartPantry maintains a dedicated automated test
suite for Mochi's AI assistant functionality.

Test file: tests/test_ai_assistant.py

The automated tests cover:

- AI response handling.
- Pantry command interpretation.
- Command validation and confirmation.
- Pantry inventory formatting.
- Supabase persistence integration.
- Storage metadata forwarding.
- Storage metadata validation.
- Date and chronology validation.
- Backward compatibility.

The test suite uses mocked dependencies without
requiring live OpenAI API calls or Supabase
database modifications.

In [29]:
import subprocess

mochi_test_run = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_ai_assistant.py",
        "-q",
        "--color=no",
    ],
    cwd=project_root,
    capture_output=True,
    text=True,
)

print(mochi_test_run.stdout)

if mochi_test_run.stderr:
    print(mochi_test_run.stderr)

assert mochi_test_run.returncode == 0, (
    "Mochi regression tests failed."
)

print("PASS: Automated Mochi regression testing.")

..............................................................           [100%]
62 passed in 0.58s

PASS: Automated Mochi regression testing.


## 7. Phase 5 Results and Conclusions

### Functional Validation Results

| Validation Category | Checks Passed | Status |
|---|---:|---|
| Storage metadata forwarding | 15/15 | PASS |
| Invalid metadata validation | 10/10 | PASS |
| Contradictory storage conditions | 8/8 | PASS |
| Date and chronology validation | 16/16 | PASS |
| Backward compatibility | 6/6 | PASS |
| **Total** | **55/55** | **PASS** |

### Automated Regression Results

| Metric | Result |
|---|---:|
| Previous automated test count | 25 |
| Phase 5 automated test count | 62 |
| Additional passing tests | 37 |
| Tests passed | 62 |
| Tests failed | 0 |
| Pass rate | 100% |

The notebook validation checks and automated pytest
results are reported separately because their
validation coverage overlaps.

### Phase 5 Accomplishments

- Integrated nine optional storage metadata fields
  into Mochi's pantry confirmation workflow.
- Preserved the existing AI command interpretation
  structure.
- Implemented metadata type and value validation.
- Implemented contradictory condition validation.
- Implemented date formatting and chronology validation.
- Prevented invalid metadata from reaching persistence.
- Preserved backward compatibility with existing
  pantry commands.
- Verified metadata forwarding using mocked persistence.
- Expanded automated regression testing from
  25 to 62 passing tests.

### Current Limitations

- Storage metadata must be supplied and confirmed
  by the user.
- Mochi does not automatically extract storage dates
  or conditions from natural-language requests.
- Missing storage information is not inferred
  or fabricated.
- Date validation is limited to the chronology
  rules currently implemented.
- Mocked notebook tests do not independently verify
  live database persistence or AI response accuracy.

### Conclusion

Phase 5 successfully extends Mochi's pantry
management functionality to support structured
storage metadata while preserving existing
command interpretation and persistence behavior.

All 55 functional validation checks and all
62 automated regression tests passed.

The implementation maintains SmartPantry's
source-based storage guidance approach without
inventing missing dates or storage conditions.

### Supporting Documentation

- modules/ai_assistant.py
- modules/supabase_pantry.py
- tests/test_ai_assistant.py
- tests/results/storage_guidance_refinement_test_results.md